In [1]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import cv2
import glob
import pandas as pd
import numpy as np

from tqdm import tqdm

from google.colab import drive

In [2]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# DATASET PATHS
# ============================================================

BASE_PATH = "/content/drive/MyDrive/Research_Project/Dataset"

ORIGA_PATH = os.path.join(BASE_PATH, "ORIGA")

PAPILA_PATH = os.path.join(BASE_PATH, "PAPILA")

RIMONE_PATH = os.path.join(BASE_PATH, "RIM-ONE-DL")

OUTPUT_CSV = "/content/drive/MyDrive/Research_Project/unified_glaucoma_dataset.csv"

In [4]:
# ============================================================
# VERIFY DATASET STRUCTURE
# ============================================================

print("="*60)
print("Checking Dataset Folder Structure")
print("="*60)

datasets = [ORIGA_PATH, PAPILA_PATH, RIMONE_PATH]

for dataset in datasets:

    if os.path.exists(dataset):

        print(f"\n✅ {os.path.basename(dataset)} Found")

        print(os.listdir(dataset))

    else:

        print(f"\n❌ {dataset} NOT FOUND")

Checking Dataset Folder Structure

✅ ORIGA Found
['dataset (divided)', 'dataset']

✅ PAPILA Found
['PapilaDB-PAPILA-17f8fa7746adb20275b5b6a0d99dc9dfe3007e9f']

✅ RIM-ONE-DL Found
['RIM-ONE_DL_images']


In [5]:
# ============================================================
# CREATE EMPTY DATASET LIST
# ============================================================

records = []

print("Dataset List Created Successfully")

Dataset List Created Successfully


In [6]:
# ============================================================
# HELPER FUNCTION
# ============================================================

IMAGE_EXTENSIONS = [
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff"
]

def is_image(filename):

    return filename.lower().endswith(tuple(IMAGE_EXTENSIONS))

In [7]:
# ============================================================
# READ ORIGA DATASET
# ============================================================

def read_origa():

    data = []

    print("\nReading ORIGA Dataset...")

    # Locate "dataset" folder automatically
    dataset_folder = None

    for root, dirs, files in os.walk(ORIGA_PATH):
        if "yes" in dirs and "no" in dirs:
            dataset_folder = root
            break

    if dataset_folder is None:
        print("❌ ORIGA dataset folder not found.")
        return data

    print("Dataset Folder:", dataset_folder)

    class_map = {
        "yes": 1,
        "no": 0
    }

    for folder_name, label in class_map.items():

        folder = os.path.join(dataset_folder, folder_name)

        image_count = 0

        for file in os.listdir(folder):

            if is_image(file):

                image_path = os.path.join(folder, file)

                data.append({

                    "image_path": image_path,
                    "label": label,
                    "dataset": "ORIGA"

                })

                image_count += 1

        print(f"{folder_name} : {image_count} images")

    print("✅ ORIGA Completed")

    return data

In [8]:
# ============================================================
# EXECUTE ORIGA READER
# ============================================================

origa_records = read_origa()

records.extend(origa_records)

print("\nTotal ORIGA Images :", len(origa_records))


Reading ORIGA Dataset...
Dataset Folder: /content/drive/MyDrive/Research_Project/Dataset/ORIGA/dataset (divided)/dataset (divided)/Train
yes : 134 images
no : 386 images
✅ ORIGA Completed

Total ORIGA Images : 520


In [9]:
# ============================================================
# VERIFY ORIGA DATA
# ============================================================

origa_df = pd.DataFrame(origa_records)

print(origa_df.head())

print("\nClass Distribution")

print(origa_df["label"].value_counts())

                                          image_path  label dataset
0  /content/drive/MyDrive/Research_Project/Datase...      1   ORIGA
1  /content/drive/MyDrive/Research_Project/Datase...      1   ORIGA
2  /content/drive/MyDrive/Research_Project/Datase...      1   ORIGA
3  /content/drive/MyDrive/Research_Project/Datase...      1   ORIGA
4  /content/drive/MyDrive/Research_Project/Datase...      1   ORIGA

Class Distribution
label
0    386
1    134
Name: count, dtype: int64


In [10]:
# ============================================================
# READ PAPILA DATASET
# ============================================================

def read_papila():

    data = []

    print("\nReading PAPILA Dataset...")

    # Locate PAPILA root automatically
    papila_root = os.path.join(PAPILA_PATH, os.listdir(PAPILA_PATH)[0])

    fundus_folder = os.path.join(papila_root, "FundusImages")
    clinical_folder = os.path.join(papila_root, "ClinicalData")

    # Read Excel files
    od = pd.read_excel(
        os.path.join(clinical_folder, "patient_data_od.xlsx"),
        header=2
    )

    os_data = pd.read_excel(
        os.path.join(clinical_folder, "patient_data_os.xlsx"),
        header=2
    )

    # Rename columns
    columns = [
        "ID","Age","Gender","Diagnosis",
        "Ref1","Ref2","Astigmatism","Pseudo",
        "IOP","Perkins","Pachymetry",
        "Axial_Length","VF_MD"
    ]

    od.columns = columns
    os_data.columns = columns

    # Add eye information
    od["Eye"] = "OD"
    os_data["Eye"] = "OS"

    papila = pd.concat([od, os_data], ignore_index=True)

    print("Total Clinical Records :", len(papila))

    # Search all fundus images once
    all_images = glob.glob(os.path.join(fundus_folder, "**", "*.jpg"), recursive=True)

    print("Total Fundus Images :", len(all_images))

    image_dict = {}

    for img in all_images:

        filename = os.path.basename(img).upper()

        image_dict[filename] = img

    added = 0

    for _, row in papila.iterrows():

        diagnosis = int(row["Diagnosis"])

        # Skip glaucoma suspect
        if diagnosis == 2:
            continue

        label = 1 if diagnosis == 1 else 0

        patient = str(row["ID"]).replace("#", "").zfill(3)

        eye = row["Eye"]

        filename = f"RET{patient}{eye}.jpg".upper()

        if filename in image_dict:

            data.append({
                "image_path": image_dict[filename],
                "label": label,
                "dataset": "PAPILA"
            })

            added += 1

    print("Matched Images :", added)

    print("✅ PAPILA Completed")

    return data

In [11]:
# ============================================================
# EXECUTE PAPILA READER
# ============================================================

papila_records = read_papila()

records.extend(papila_records)

print("Total PAPILA Images :", len(papila_records))


Reading PAPILA Dataset...
Total Clinical Records : 488
Total Fundus Images : 488
Matched Images : 420
✅ PAPILA Completed
Total PAPILA Images : 420


In [12]:
# ============================================================
# VERIFY PAPILA
# ============================================================

papila_df = pd.DataFrame(papila_records)

print(papila_df.head())

print("\nClass Distribution")

print(papila_df["label"].value_counts())

                                          image_path  label dataset
0  /content/drive/MyDrive/Research_Project/Datase...      1  PAPILA
1  /content/drive/MyDrive/Research_Project/Datase...      1  PAPILA
2  /content/drive/MyDrive/Research_Project/Datase...      1  PAPILA
3  /content/drive/MyDrive/Research_Project/Datase...      1  PAPILA
4  /content/drive/MyDrive/Research_Project/Datase...      1  PAPILA

Class Distribution
label
0    333
1     87
Name: count, dtype: int64


In [13]:
# ============================================================
# READ RIM-ONE DL DATASET
# ============================================================

def read_rimone():

    data = []

    print("\nReading RIM-ONE DL Dataset...")

    # Automatically locate partitioned_randomly folder
    partition_root = None

    for root, dirs, files in os.walk(RIMONE_PATH):
        if "training_set" in dirs and "test_set" in dirs:
            partition_root = root
            break

    if partition_root is None:
        print("❌ partitioned_randomly folder not found.")
        return data

    print("Partition Folder:", partition_root)

    subsets = ["training_set", "test_set"]

    classes = {
        "glaucoma": 1,
        "normal": 0
    }

    total_images = 0

    for subset in subsets:

        subset_path = os.path.join(partition_root, subset)

        if not os.path.exists(subset_path):
            continue

        print(f"\n{subset}")

        for class_name, label in classes.items():

            class_path = os.path.join(subset_path, class_name)

            if not os.path.exists(class_path):
                continue

            count = 0

            for file in os.listdir(class_path):

                if is_image(file):

                    data.append({
                        "image_path": os.path.join(class_path, file),
                        "label": label,
                        "dataset": "RIM-ONE DL"
                    })

                    count += 1
                    total_images += 1

            print(f"{class_name} : {count}")

    print("\nTotal Images :", total_images)

    print("✅ RIM-ONE Completed")

    return data

In [14]:
# ============================================================
# EXECUTE RIM-ONE READER
# ============================================================

rimone_records = read_rimone()

records.extend(rimone_records)

print("Total RIM-ONE Images :", len(rimone_records))


Reading RIM-ONE DL Dataset...
Partition Folder: /content/drive/MyDrive/Research_Project/Dataset/RIM-ONE-DL/RIM-ONE_DL_images/partitioned_randomly

training_set
glaucoma : 120
normal : 219

test_set
glaucoma : 52
normal : 94

Total Images : 485
✅ RIM-ONE Completed
Total RIM-ONE Images : 485


In [15]:
rimone_df = pd.DataFrame(rimone_records)

print(rimone_df.head())

print("\nClass Distribution")

print(rimone_df["label"].value_counts())

                                          image_path  label     dataset
0  /content/drive/MyDrive/Research_Project/Datase...      1  RIM-ONE DL
1  /content/drive/MyDrive/Research_Project/Datase...      1  RIM-ONE DL
2  /content/drive/MyDrive/Research_Project/Datase...      1  RIM-ONE DL
3  /content/drive/MyDrive/Research_Project/Datase...      1  RIM-ONE DL
4  /content/drive/MyDrive/Research_Project/Datase...      1  RIM-ONE DL

Class Distribution
label
0    313
1    172
Name: count, dtype: int64


In [16]:
# ============================================================
# CREATE FINAL DATAFRAME
# ============================================================

df = pd.DataFrame(records)

print("="*60)
print("UNIFIED DATASET CREATED")
print("="*60)

print("Total Images :", len(df))

print("\nDataset Distribution")

print(df["dataset"].value_counts())

print("\nClass Distribution")

print(df["label"].value_counts())

UNIFIED DATASET CREATED
Total Images : 1425

Dataset Distribution
dataset
ORIGA         520
RIM-ONE DL    485
PAPILA        420
Name: count, dtype: int64

Class Distribution
label
0    1032
1     393
Name: count, dtype: int64


In [17]:
# ============================================================
# REMOVE DUPLICATES
# ============================================================

duplicates = df.duplicated(subset=["image_path"])

print("Duplicate Images :", duplicates.sum())

df = df.drop_duplicates(subset=["image_path"])

print("Remaining Images :", len(df))

Duplicate Images : 0
Remaining Images : 1425


In [20]:
# ============================================================
# ADD EXTRA COLUMNS
# ============================================================

df["image_name"] = df["image_path"].apply(os.path.basename)

df["label_name"] = df["label"].map({
    0: "Normal",
    1: "Glaucoma"
})

# Arrange columns
df = df[
    [
        "image_name",
        "image_path",
        "label",
        "label_name",
        "dataset"
    ]
]

print(df.head())

  image_name                                         image_path  label  \
0    072.jpg  /content/drive/MyDrive/Research_Project/Datase...      1   
1    086.jpg  /content/drive/MyDrive/Research_Project/Datase...      1   
2    036.jpg  /content/drive/MyDrive/Research_Project/Datase...      1   
3    076.jpg  /content/drive/MyDrive/Research_Project/Datase...      1   
4    087.jpg  /content/drive/MyDrive/Research_Project/Datase...      1   

  label_name dataset  
0   Glaucoma   ORIGA  
1   Glaucoma   ORIGA  
2   Glaucoma   ORIGA  
3   Glaucoma   ORIGA  
4   Glaucoma   ORIGA  


In [21]:
# ============================================================
# SAVE CSV
# ============================================================

OUTPUT_CSV = "/content/drive/MyDrive/Research_Project/unified_glaucoma_dataset.csv"

df.to_csv(OUTPUT_CSV, index=False)

print("✅ CSV Saved Successfully")
print("Location:", OUTPUT_CSV)

✅ CSV Saved Successfully
Location: /content/drive/MyDrive/Research_Project/unified_glaucoma_dataset.csv


In [19]:
print(df["dataset"].value_counts())

dataset
ORIGA         520
RIM-ONE DL    485
PAPILA        420
Name: count, dtype: int64
